In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex
from sentence_transformers import SentenceTransformer
import citation_utils
import metric_utils
import reranker_utils
import hits_utils

/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

# model = SentenceTransformer("/root/.cache/modelscope/hub/models/Qwen/Qwen3-Embedding-0___6B", model_kwargs={"torch_dtype": "float16"})

# dense_index = DenseIndex(model, "../data/processed/_dense_court", court_doc)
dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
sparse_index.load()

# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True, show_progress_bar=False)

DenseIndex.embeddings:  (2107648, 1024)


In [5]:
from tqdm.notebook import tqdm
import citation_utils2
import text_chunk

hits_l = []
gold_citations_l = []

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

for query, gold_citations in tqdm(zip(valid_df['query2'].tolist(), valid_df['gold_citations']), total=len(valid_df)):
    hits1 = dense_index.search_with_score(query, top_k=1000)
    hits2 = sparse_index.search_with_score(query, top_k=1000)
    # print(hits1[0], hits2[0])
    hits_merge = hits_utils.merge_hits_with_score_l_by_weighted_add(hits1, hits2, 0.2, 0.8)
    hits = [hit for hit,score in hits_merge]

    sentences = []
    for hit in hits:
        s_l = citation_utils.split_sentences(hit['text'])
        s_l_2 = text_chunk.sliding_window_for_sentences(s_l, 3, 2)
        for s in s_l_2:
            citations = citation_utils.extract_citations_from_text(s)
            if len(citations) > 0:
                sentences.append(s)

    print("sentences.len:", len(sentences))
    
    gold_citations_l.append(gold_citations.split(";"))

    sentence_with_score_l = reranker_utils.rerank_by_dense_batch(reranker, query, sentences, top_k=200, batch_size=10)

    print("sentence_score.len:", len(sentence_with_score_l))
    
    citation_score_d = {}
    for sentence,score in sentence_with_score_l:
        citations = citation_utils.extract_citations_from_text(sentence)
        normalized_citations = []
        for c in citations:
            parsed = citation_utils2.parse_citation(c)
            if parsed is not None:
                normalized_citations.append(citation_utils2.normalize(parsed))
        for c in normalized_citations:
            citation_score_d[c] = citation_score_d.get(c,0) + score
    
    hits = []
    for citation,score in citation_score_d.items():
        hits.append((citation, score))

    sorted_hits = [(hit,score) for hit, score in sorted(hits, key=lambda x: x[1], reverse=True)]

    # dedup
    dedup_sorted_hits = citation_utils2.dedup_with_score(sorted_hits, citation_utils2.parse_citation, citation_utils2.normalize)

    print('hits.len:', len(sorted_hits), 'dedup_sorted_hits.len:', len(dedup_sorted_hits))

    hits = []
    for citation,score in dedup_sorted_hits:
        hits.append({'citation':citation})

    hits_l.append(hits)

  0%|          | 0/10 [00:00<?, ?it/s]

sentences.len: 5213
sentence_score.len: 200
hits.len: 27 dedup_sorted_hits.len: 21
sentences.len: 4805
sentence_score.len: 200
hits.len: 73 dedup_sorted_hits.len: 55
sentences.len: 7758
sentence_score.len: 200
hits.len: 34 dedup_sorted_hits.len: 30
sentences.len: 6667
sentence_score.len: 200
hits.len: 133 dedup_sorted_hits.len: 122
sentences.len: 10594
sentence_score.len: 200
hits.len: 98 dedup_sorted_hits.len: 84
sentences.len: 9336
sentence_score.len: 200
hits.len: 148 dedup_sorted_hits.len: 131
sentences.len: 11593
sentence_score.len: 200
hits.len: 167 dedup_sorted_hits.len: 138
sentences.len: 11527
sentence_score.len: 200
hits.len: 122 dedup_sorted_hits.len: 114
sentences.len: 6265
sentence_score.len: 200
hits.len: 134 dedup_sorted_hits.len: 109
sentences.len: 10241
sentence_score.len: 200
hits.len: 146 dedup_sorted_hits.len: 131


In [6]:
for limit in [5,15,20,25,30,35,40,45]:
    r = metric_utils.cal_recall(hits_l, gold_citations_l, lambda x: limit)
    p = metric_utils.cal_precision(hits_l, gold_citations_l, lambda x: limit)
    f1 = metric_utils.cal_f1(r,p)
    print(f"[{limit}] r:",r, "p:",p, "f1:",f1)

[5] r: 0.08901187376721222 p: 0.32 f1: 0.1392810401475991
[15] r: 0.1348148920624111 p: 0.17333333333333334 f1: 0.15166671554989894
[20] r: 0.14390580115332016 p: 0.13500000000000004 f1: 0.1393107140501438
[25] r: 0.14946135670887573 p: 0.1182857142857143 f1: 0.132058537714338
[30] r: 0.16028007015916812 p: 0.10761904761904761 f1: 0.1287737611523141
[35] r: 0.16742292730202527 p: 0.0980952380952381 f1: 0.12370823586944821
[40] r: 0.1745657844448824 p: 0.09095238095238095 f1: 0.11959387941933862
[45] r: 0.18134356222266018 p: 0.08761904761904762 f1: 0.11815136849799462
